# 01 — Exploratory data analysis

Purpose: establish, from the data itself, the constants the engine depends on.
Nothing below may be hardcoded downstream — `risk_engine` recomputes it, this
notebook is where the choice is *justified*.

Questions this notebook must answer:

1. **Observation window** — first and last telemetry timestamp, observed days,
   and therefore the annualization factor `365 / observed_days`.
2. **Cross-feed overlap** — how many events appear in both SIEM and EDR on the
   same (asset, MITRE technique, timestamp), and what merging without dedup
   would do to frequency.
3. **EDR cut points** — where to slice `risk` (0–999) so it maps onto the SIEM
   Low/Medium/High/Critical classes. The chosen thresholds become named
   constants in `risk_engine.ingestion`, with a comment pointing back here.
4. **Incident base hygiene** — mojibake sector duplicates, and the `-1`
   sentinel in `financial_loss_eur` (missing, never €0).
5. **Loss tail** — is a lognormal fit on the logs defensible? Check with QQ and
   KS before accepting it, and look at what the top decile does to the mean.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path.cwd().parent / "data"
sorted(p.name for p in DATA_DIR.glob("*.csv"))